In [ ]:
!pip install gradio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import gradio as gr
import matplotlib.pyplot as plt
import re
import numpy as np
from datetime import datetime

print("✅ Libraries imported successfully!")

# Load the processed data from your backend
csv_path = '/content/drive/MyDrive/email_dataset/parsed_df.csv'
pickle_path = '/content/drive/MyDrive/email_dataset/parsed_df.pkl'

try:
    # Try loading pickle first (preserves data types better)
    if pd.io.common.file_exists(pickle_path):
        parsed_df = pd.read_pickle(pickle_path)
        print(f"✅ Loaded data from pickle: {len(parsed_df)} emails")
    elif pd.io.common.file_exists(csv_path):
        parsed_df = pd.read_csv(csv_path)
        print(f"✅ Loaded data from CSV: {len(parsed_df)} emails")
    else:
        raise FileNotFoundError("No saved data found")

    print(f"📊 Data shape: {parsed_df.shape}")
    print(f"📋 Columns: {parsed_df.columns.tolist()}")
    print(f"📈 Categories: {parsed_df['Category'].unique() if 'Category' in parsed_df.columns else 'No Category column'}")

except Exception as e:
    print(f"⚠️ Error loading data: {e}")
    print("Creating sample data for testing...")

    # Sample data as fallback
    sample_data = {
        'From': ['john.doe@company.com', 'finance@billing.com', 'sarah@personal.com',
                 'team@project.com', 'newsletter@tech.com', 'manager@work.com',
                 'alert@security.com', 'invites@events.com', 'support@service.com',
                 'hr@company.com'],
        'Subject': ['Meeting scheduled for tomorrow', 'Invoice #12345 payment due',
                   'Birthday party invitation', 'Project deadline extended',
                   'Weekly tech newsletter', 'Performance review meeting',
                   'Security alert: unusual login', 'Conference invitation',
                   'Your subscription invoice', 'Team building event'],
        'Category': ['Work', 'Finance', 'Personal', 'Work', 'Other',
                    'Work', 'Other', 'Personal', 'Finance', 'Personal'],
        'Agent_Action': ['📅 Add to calendar', '💰 Forward to accounts', '💝 Mark as personal',
                        '📅 Add to calendar', '📁 Archive', '📅 Add to calendar',
                        '📁 Archive', '💝 Mark as personal', '💰 Forward to accounts', '💝 Mark as personal']
    }
    parsed_df = pd.DataFrame(sample_data)
    print(f"✅ Created {len(parsed_df)} sample emails for testing")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Libraries imported successfully!
✅ Loaded data from pickle: 517401 emails
📊 Data shape: (517401, 5)
📋 Columns: ['Message-ID', 'From', 'Subject', 'Category', 'Agent_Action']
📈 Categories: ['Other' 'Work' 'Finance' 'Personal']


In [ ]:
def categorize(subject):
    """Categorize email based on subject line"""
    if subject is None or pd.isna(subject):
        return "Unknown"
    subject = subject.lower()
    if "meeting" in subject or "schedule" in subject or "deadline" in subject:
        return "Work"
    elif "invoice" in subject or "payment" in subject or "bill" in subject:
        return "Finance"
    elif "party" in subject or "invitation" in subject or "birthday" in subject:
        return "Personal"
    else:
        return "Other"

def agent_action(row):
    """Determine action based on category"""
    if row["Category"] == "Work":
        return "📅 Add to calendar / notify team"
    elif row["Category"] == "Finance":
        return "💰 Forward to accounts department"
    elif row["Category"] == "Personal":
        return "💝 Mark as personal / no action"
    else:
        return "📁 Archive"

def parse_email_from_text(raw_text):
    """Parse email text to extract headers"""
    headers = {}
    if pd.isna(raw_text):
        return headers
    for line in str(raw_text).split("\n"):
        if line.startswith("Message-ID:"):
            headers["Message-ID"] = line.replace("Message-ID:", "").strip()
        elif line.startswith("From:"):
            headers["From"] = line.replace("From:", "").strip()
        elif line.startswith("Subject:"):
            headers["Subject"] = line.replace("Subject:", "").strip()
    return headers


In [ ]:

def categorize_new_email(subject, from_email, message_body=""):
    """Categorize a single new email"""
    if not subject or subject.strip() == "":
        return "⚠️ Please enter an email subject"

    category = categorize(subject)
    action = agent_action(pd.Series({"Category": category}))

    # Color coding
    color_map = {
        "Work": "🔵",
        "Finance": "🟢",
        "Personal": "🟠",
        "Other": "⚪",
        "Unknown": "⚫"
    }

    result = f"""
<div style='padding: 20px; border-radius: 10px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white;'>
<h2>📧 Email Analysis Results</h2>
<hr>
<p><strong>📨 From:</strong> {from_email}</p>
<p><strong>📌 Subject:</strong> {subject}</p>
<p><strong>🏷️ Category:</strong> {color_map.get(category, '📌')} <strong>{category}</strong></p>
<p><strong>🎯 Recommended Action:</strong> {action}</p>
</div>

<div style='margin-top: 20px; padding: 15px; background-color: #f0f0f0; border-radius: 10px;'>
<h3>🔍 Analysis Details:</h3>
<ul>
<li>{"✅ Work-related keywords found (meeting/schedule/deadline)" if category == "Work" else "❌ No work keywords"}</li>
<li>{"✅ Financial keywords found (invoice/payment/bill)" if category == "Finance" else "❌ No finance keywords"}</li>
<li>{"✅ Personal keywords found (party/invitation/birthday)" if category == "Personal" else "❌ No personal keywords"}</li>
<li>{"📌 Categorized as Other - no specific rules matched" if category == "Other" else ""}</li>
</ul>
</div>

<div style='margin-top: 20px; padding: 15px; background-color: #e3f2fd; border-radius: 10px;'>
<h3>💡 Next Steps:</h3>
<p>{action}</p>
</div>
"""
    return result

def get_statistics():
    """Get comprehensive statistics"""
    stats = {
        "Total Emails": len(parsed_df),
        "Categories": parsed_df['Category'].nunique() if 'Category' in parsed_df.columns else 0,
        "Unique Senders": parsed_df['From'].nunique() if 'From' in parsed_df.columns else 0,
    }

    # Category distribution
    if 'Category' in parsed_df.columns:
        category_counts = parsed_df['Category'].value_counts()
        stats['Most Common Category'] = category_counts.index[0] if len(category_counts) > 0 else "N/A"
        stats['Least Common Category'] = category_counts.index[-1] if len(category_counts) > 0 else "N/A"
    else:
        stats['Most Common Category'] = "N/A"
        stats['Least Common Category'] = "N/A"

    return stats

def show_dashboard():
    """Display comprehensive analytics dashboard"""
    if 'Category' not in parsed_df.columns:
        # Create dummy data if category column missing
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.text(0.5, 0.5, 'No category data available\nRun backend processing first',
                ha='center', va='center', fontsize=16)
        ax.axis('off')
        return fig, pd.DataFrame()

    # Create figure with subplots
    fig = plt.figure(figsize=(16, 10))

    # 1. Category Distribution (Pie Chart)
    ax1 = plt.subplot(2, 3, 1)
    category_counts = parsed_df["Category"].value_counts()
    colors_pie = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
    wedges, texts, autotexts = ax1.pie(category_counts.values,
                                        labels=category_counts.index,
                                        autopct='%1.1f%%',
                                        colors=colors_pie[:len(category_counts)],
                                        explode=[0.05]*len(category_counts))
    ax1.set_title('Email Categories Distribution', fontsize=14, fontweight='bold')

    # 2. Action Distribution (Bar Chart)
    ax2 = plt.subplot(2, 3, 2)
    action_counts = parsed_df["Agent_Action"].value_counts()
    bars = ax2.bar(range(len(action_counts)), action_counts.values,
                   color=['#4CAF50', '#2196F3', '#FF9800', '#9C27B0'][:len(action_counts)])
    ax2.set_xticks(range(len(action_counts)))
    ax2.set_xticklabels([a.split(' ')[0] for a in action_counts.index],
                        rotation=45, ha='right', fontsize=10)
    ax2.set_title('Recommended Actions Distribution', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Number of Emails', fontsize=12)

    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, action_counts.values)):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(val), ha='center', va='bottom', fontweight='bold')

    # 3. Top Senders (Horizontal Bar Chart)
    ax3 = plt.subplot(2, 3, 3)
    if 'From' in parsed_df.columns:
        top_senders = parsed_df['From'].value_counts().head(10)
        senders_short = [s[:25] + '...' if len(s) > 25 else s for s in top_senders.index]
        ax3.barh(range(len(top_senders)), top_senders.values, color='skyblue')
        ax3.set_yticks(range(len(top_senders)))
        ax3.set_yticklabels(senders_short, fontsize=9)
        ax3.set_xlabel('Number of Emails', fontsize=12)
        ax3.set_title('Top 10 Senders', fontsize=14, fontweight='bold')
        ax3.invert_yaxis()

    # 4. Category Trend (Line Chart)
    ax4 = plt.subplot(2, 3, 4)
    if 'Category' in parsed_df.columns:
        # Simulate trend data (if no date column, use index)
        category_trend = parsed_df['Category'].value_counts().sort_index()
        ax4.plot(range(len(category_trend)), category_trend.values,
                marker='o', linewidth=2, markersize=8, color='purple')
        ax4.set_xticks(range(len(category_trend)))
        ax4.set_xticklabels(category_trend.index, rotation=45, ha='right')
        ax4.set_ylabel('Count', fontsize=12)
        ax4.set_title('Email Volume by Category', fontsize=14, fontweight='bold')
        ax4.grid(True, alpha=0.3)

    # 5. Summary Statistics Table
    ax5 = plt.subplot(2, 3, 5)
    ax5.axis('tight')
    ax5.axis('off')

    stats_data = [
        ['Total Emails', len(parsed_df)],
        ['Categories', parsed_df['Category'].nunique()],
        ['Unique Senders', parsed_df['From'].nunique() if 'From' in parsed_df.columns else 0],
        ['Most Common', parsed_df['Category'].mode()[0] if len(parsed_df['Category'].mode()) > 0 else 'N/A'],
        ['Work Emails', (parsed_df['Category'] == 'Work').sum()],
        ['Finance Emails', (parsed_df['Category'] == 'Finance').sum()],
        ['Personal Emails', (parsed_df['Category'] == 'Personal').sum()],
        ['Other Emails', (parsed_df['Category'] == 'Other').sum()],
    ]

    table = ax5.table(cellText=stats_data, colLabels=['Metric', 'Value'],
                     cellLoc='left', loc='center',
                     colWidths=[0.5, 0.3])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)

    # Color the header
    for i in range(2):
        table[(0, i)].set_facecolor('#667eea')
        table[(0, i)].set_text_props(weight='bold', color='white')

    ax5.set_title('Summary Statistics', fontsize=14, fontweight='bold', pad=20)

    # 6. Category Donut Chart (Alternative View)
    ax6 = plt.subplot(2, 3, 6)
    wedges, texts, autotexts = ax6.pie(category_counts.values,
                                        labels=category_counts.index,
                                        autopct='%1.1f%%',
                                        colors=colors_pie[:len(category_counts)],
                                        wedgeprops=dict(width=0.6))
    ax6.set_title('Category Distribution (Donut View)', fontsize=14, fontweight='bold')

    plt.suptitle('Email Analytics Dashboard', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()

    # Get sample emails for table
    sample_emails = parsed_df[['From', 'Subject', 'Category', 'Agent_Action']].head(15)

    return fig, sample_emails

def search_emails(keyword, category_filter, sender_filter=""):
    """Search emails with multiple filters"""
    filtered_df = parsed_df.copy()

    # Apply keyword search
    if keyword and keyword.strip():
        keyword = keyword.lower()
        filtered_df = filtered_df[
            filtered_df['Subject'].str.lower().str.contains(keyword, na=False) |
            filtered_df['From'].str.lower().str.contains(keyword, na=False)
        ]

    # Apply category filter
    if category_filter != "All":
        filtered_df = filtered_df[filtered_df['Category'] == category_filter]

    # Apply sender filter
    if sender_filter and sender_filter.strip():
        sender_filter = sender_filter.lower()
        filtered_df = filtered_df[filtered_df['From'].str.lower().str.contains(sender_filter, na=False)]

    if len(filtered_df) == 0:
        return pd.DataFrame(columns=['From', 'Subject', 'Category', 'Agent_Action']), "No emails found matching your criteria."

    result_df = filtered_df[['From', 'Subject', 'Category', 'Agent_Action']].head(50)
    return result_df, f"Found {len(filtered_df)} emails (showing first 50)"

def export_filtered_emails(keyword, category_filter, sender_filter=""):
    """Export filtered emails to CSV"""
    filtered_df, _ = search_emails(keyword, category_filter, sender_filter)

    if len(filtered_df) > 0:
        export_path = f'/content/drive/MyDrive/email_dataset/exported_emails_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
        filtered_df.to_csv(export_path, index=False)
        return f"✅ Exported {len(filtered_df)} emails to: {export_path}"
    else:
        return "⚠️ No emails to export"

def get_unique_senders():
    """Get list of unique senders for dropdown"""
    if 'From' in parsed_df.columns:
        senders = sorted(parsed_df['From'].unique())
        return gr.Dropdown(choices=senders, label="Filter by Sender")
    return gr.Dropdown(choices=[], label="Filter by Sender")

In [ ]:
# Custom CSS for better UI
custom_css = """
.gradio-container {
    max-width: 1400px !important;
    margin: auto !important;
}
.header {
    text-align: center;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 20px;
    border-radius: 10px;
    color: white;
    margin-bottom: 20px;
}
.stats-card {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 15px;
    border-radius: 10px;
    color: white;
    text-align: center;
    margin: 5px;
}
"""

# Get statistics
stats = get_statistics()

# Create the interface
with gr.Blocks(title="Smart Email Assistant", theme=gr.themes.Soft(), css=custom_css) as demo:

    # Header
    gr.Markdown("""
    <div class="header">
    <h1>📧 Smart Email Assistant</h1>
    <h3>AI-Powered Email Management System with LangGraph Architecture</h3>
    </div>
    """)

    # Statistics Cards
    with gr.Row():
        with gr.Column():
            gr.Markdown(f"""
            <div class="stats-card">
            <h3>📊 Total Emails</h3>
            <h2>{stats['Total Emails']}</h2>
            </div>
            """)
        with gr.Column():
            gr.Markdown(f"""
            <div class="stats-card">
            <h3>📂 Categories</h3>
            <h2>{stats['Categories']}</h2>
            </div>
            """)
        with gr.Column():
            gr.Markdown(f"""
            <div class="stats-card">
            <h3>👤 Unique Senders</h3>
            <h2>{stats['Unique Senders']}</h2>
            </div>
            """)
        with gr.Column():
            gr.Markdown(f"""
            <div class="stats-card">
            <h3>🏆 Most Common</h3>
            <h2>{stats['Most Common Category']}</h2>
            </div>
            """)

    # Main Tabs
    with gr.Tabs():
        # Tab 1: Categorize New Email
        with gr.TabItem("📝 Categorize New Email", id="categorize"):
            with gr.Row():
                with gr.Column(scale=1):
                    subject_input = gr.Textbox(
                        label="📌 Email Subject",
                        placeholder="Enter the email subject...",
                        lines=2
                    )
                    from_input = gr.Textbox(
                        label="👤 From Email",
                        placeholder="sender@example.com"
                    )
                    body_input = gr.Textbox(
                        label="📄 Email Body (Optional)",
                        placeholder="Enter email content...",
                        lines=5
                    )
                    analyze_btn = gr.Button("🔍 Analyze Email", variant="primary", size="lg")

                with gr.Column(scale=1):
                    result_output = gr.Markdown(label="📊 Analysis Result")

            # Example emails
            gr.Markdown("### 📋 Try These Examples:")
            gr.Examples(
                examples=[
                    ["Meeting scheduled for tomorrow at 10 AM", "manager@company.com", "Please join the team meeting"],
                    ["Invoice #12345 payment due on Friday", "billing@company.com", "Your invoice is ready"],
                    ["You're invited to my birthday party!", "friend@personal.com", "Hope you can make it!"],
                    ["Weekly report attached", "colleague@work.com", "Please review"],
                    ["Project deadline extended to next month", "project@work.com", "New timeline announced"],
                    ["Your subscription payment failed", "billing@service.com", "Please update payment method"],
                ],
                inputs=[subject_input, from_input, body_input]
            )

            analyze_btn.click(
                categorize_new_email,
                inputs=[subject_input, from_input, body_input],
                outputs=[result_output]
            )

        # Tab 2: Analytics Dashboard
        with gr.TabItem("📊 Analytics Dashboard", id="dashboard"):
            dashboard_btn = gr.Button("🔄 Refresh Dashboard", variant="secondary", size="lg")
            with gr.Row():
                dashboard_plot = gr.Plot(label="Email Analytics Dashboard")
            with gr.Row():
                sample_table = gr.Dataframe(
                    label="📧 Recent Emails Sample",
                    headers=["From", "Subject", "Category", "Action"],
                    interactive=False,
                    max_height=400
                )

            dashboard_btn.click(show_dashboard, outputs=[dashboard_plot, sample_table])
            demo.load(show_dashboard, outputs=[dashboard_plot, sample_table])

        # Tab 3: Advanced Search
        with gr.TabItem("🔍 Advanced Search", id="search"):
            with gr.Row():
                search_keyword = gr.Textbox(
                    label="🔎 Search Keyword",
                    placeholder="Enter keyword to search in subject or sender...",
                    scale=2
                )
                category_filter = gr.Dropdown(
                    choices=["All", "Work", "Finance", "Personal", "Other", "Unknown"],
                    value="All",
                    label="📂 Filter by Category",
                    scale=1
                )
                sender_filter = gr.Textbox(
                    label="👤 Filter by Sender",
                    placeholder="Enter sender email...",
                    scale=1
                )

            with gr.Row():
                search_btn = gr.Button("🔍 Search", variant="primary", scale=1)
                export_btn = gr.Button("📥 Export Results", variant="secondary", scale=1)

            search_status = gr.Markdown("")
            search_results = gr.Dataframe(
                label="📧 Search Results",
                headers=["From", "Subject", "Category", "Action"],
                interactive=True,
                max_height=500
            )

            search_btn.click(
                search_emails,
                inputs=[search_keyword, category_filter, sender_filter],
                outputs=[search_results, search_status]
            )

            export_btn.click(
                export_filtered_emails,
                inputs=[search_keyword, category_filter, sender_filter],
                outputs=[search_status]
            )

        # Tab 4: Data Overview
        with gr.TabItem("📋 Data Overview", id="data"):
            view_all_btn = gr.Button("📋 View All Emails", variant="secondary")
            all_emails_table = gr.Dataframe(
                label="Complete Email Dataset",
                headers=["From", "Subject", "Category", "Action"],
                interactive=True,
                max_height=600
            )

            view_all_btn.click(
                lambda: parsed_df[['From', 'Subject', 'Category', 'Agent_Action']],
                outputs=[all_emails_table]
            )

            # Show data info
            gr.Markdown(f"""
            ### 📊 Dataset Information
            - **Total Records:** {len(parsed_df)}
            - **Columns:** {', '.join(parsed_df.columns)}
            - **Date Range:** Loaded from your backend processing
            - **Categories:** {', '.join(parsed_df['Category'].unique()) if 'Category' in parsed_df.columns else 'N/A'}
            """)



/tmp/ipykernel_8550/3096383807.py:29: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Smart Email Assistant", theme=gr.themes.Soft(), css=custom_css) as demo:
/tmp/ipykernel_8550/3096383807.py:29: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="Smart Email Assistant", theme=gr.themes.Soft(), css=custom_css) as demo:


In [ ]:
print("\n" + "="*50)
print("🚀 Launching Smart Email Assistant...")
print("="*50)
print(f"📊 Loaded {len(parsed_df)} emails from backend")
print(f"📂 Categories available: {parsed_df['Category'].unique() if 'Category' in parsed_df.columns else 'N/A'}")
print("\n📱 The app will be available at the URL below")
print("🔗 Click on the Gradio.live URL to open the interface")
print("⚠️ Keep this tab open while using the app")
print("="*50 + "\n")

# Launch with share=True for public URL
demo.launch(share=True, debug=True)



🚀 Launching Smart Email Assistant...
📊 Loaded 517401 emails from backend
📂 Categories available: ['Other' 'Work' 'Finance' 'Personal']

📱 The app will be available at the URL below
🔗 Click on the Gradio.live URL to open the interface
⚠️ Keep this tab open while using the app

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://386f13d8fb0700a1ab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://386f13d8fb0700a1ab.gradio.live
